In [7]:
import pandas as pd

In [8]:
# Check available sheet names first
xls = pd.ExcelFile('../data/online_retail_II.xlsx')
print(xls.sheet_names)

['Year 2009-2010', 'Year 2010-2011']


In [9]:
# Load each sheet into its own DataFrame
df_2009_2010 = pd.read_excel(xls, sheet_name='Year 2009-2010')
df_2010_2011 = pd.read_excel(xls, sheet_name='Year 2010-2011')

In [10]:
df_2009_2010["SourceSheet"] = "2009-2010"
df_2010_2011["SourceSheet"] = "2010-2011"

In [11]:
# Combine into one DataFrame
df_all = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)
df = df_all.copy()

In [12]:
df.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010


In [13]:
df.isna().sum() 

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
SourceSheet         0
dtype: int64

In [14]:
# Let's check how big dataset is.
# .shape gives us (number of rows, number of columns).
print("The dataset has:", df.shape[0], "rows and", df.shape[1], "columns")

The dataset has: 1067371 rows and 9 columns


In [15]:

# List the column names so we know exactly what we have.
print("\nColumns:", list(df.columns))


Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'SourceSheet']


In [16]:
# Information in dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
 8   SourceSheet  1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(2)
memory usage: 96.2+ MB


## Data Cleaning

### 1. Fix data types

In [17]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Invoice'] = df['Invoice'].astype(str).str.strip()
df['StockCode'] = df['StockCode'].astype(str).str.strip().str.upper()
df['Description'] = df['Description'].astype(str).str.strip()
df['Country'] = df['Country'].astype(str).str.strip()

### 2. Remove exact duplicate rows

In [18]:
# Show the duplicated rows (all occurrences, not just the extra copies)
duplicate_rows = df[df.duplicated(subset=list(df.columns), keep=False)]
print(f"Found {len(duplicate_rows)} rows involved in duplication")
duplicate_rows.sort_values(by=list(df.columns)).head(10)

Found 23430 rows involved in duplication


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,2009-2010
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,2009-2010
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,2009-2010


In [19]:
# 2. Remove exact duplicate rows
before = len(df)
df_clean = df.drop_duplicates(list(df.columns))
print(f"Dropped {before - len(df_clean)} exact duplicate rows -> {len(df_clean)} left")

Dropped 12133 exact duplicate rows -> 1055238 left


### 3. Drop rows with missing Customer ID

In [20]:
before = len(df_clean)
df_clean = df_clean.dropna(subset=['Customer ID'])
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)
print(f"Dropped {before - len(df_clean)} rows with missing Customer ID -> {len(df_clean)} left")

Dropped 242870 rows with missing Customer ID -> 812368 left


### 4. Drop rows with missing Description

In [21]:
before = len(df_clean)
df_clean = df_clean.dropna(subset=['Description'])
print(f"Dropped {before - len(df_clean)} rows with missing Description -> {len(df_clean)} left")

Dropped 0 rows with missing Description -> 812368 left


### 5. Remove non-product / administrative stock codes

In [22]:
# -----------------------------------------------------------------------
# 5. Remove non-product / administrative stock codes
# -----------------------------------------------------------------------
# These codes are postage, fees, manual adjustments, bank charges, samples,
# discounts and test entries — not real products, so they'd distort RFM,
# segmentation and product-level analysis.
admin_codes = [
    'POST', 'DOT', 'M', 'C2', 'D', 'S', 'BANK CHARGES',
    'ADJUST', 'ADJUST2', 'AMAZONFEE', 'CRUK', 'TEST001', 'TEST002', 'B'
]
before = len(df_clean)
df_clean = df_clean[~df_clean['StockCode'].isin(admin_codes)]
print(f"Dropped {before - len(df_clean)} rows with admin/non-product stock codes -> {len(df_clean)} left")


Dropped 3706 rows with admin/non-product stock codes -> 808662 left


### 6. Separate cancellations (returns) from the sales dataset

Invoices starting with 'C' are cancellations. Keep them in a separate  dataframe in case you want to analyze returns later,
but exclude them  from the "sales" dataset used for RFM/segmentation/prediction.

In [23]:
is_cancelled = df_clean['Invoice'].str.startswith('C')
df_returns = df_clean[is_cancelled].copy()
df_clean = df_clean[~is_cancelled].copy()
print(f"Separated {len(df_returns)} cancelled-order rows into df_returns -> {len(df_clean)} sales rows left")

Separated 17879 cancelled-order rows into df_returns -> 790783 sales rows left


### 7. Remove non-positive quantities and prices
A handful of negative-quantity rows aren't flagged as cancellations , and  zero/negative prices are data errors 
or free/adjustment items with no real revenue signal.

In [24]:
before = len(df_clean)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]
print(f"Dropped {before - len(df_clean)} rows with Quantity <= 0 or Price <= 0 -> {len(df_clean)} left")

Dropped 62 rows with Quantity <= 0 or Price <= 0 -> 790721 left


In [25]:
# -----------------------------------------------------------------------
# 8. Remove extreme price outliers (optional sanity cap)
# -----------------------------------------------------------------------
# A few Price values are in the thousands and clearly not per-unit retail  prices.
#Flag anything beyond the 99.9th percentile for review instead of  silently dropping, so you can decide case by case.
#price_cap = df_clean['Price'].quantile(0.999)
#n_outliers = (df_clean['Price'] > price_cap).sum()
#print(f"{n_outliers} rows have Price above the 99.9th percentile ({price_cap:.2f}) — inspect before deciding to cap/drop")

### 9. Add the Revenue column (needed for EDA/RFM downstream)

In [26]:
df_clean= df_clean.drop(columns=['SourceSheet'])

In [27]:
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']
df_clean.head(2)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0


### Final Data cleaning Summary

In [28]:
df_clean = df_clean.reset_index(drop=True)
print("\n==== CLEANING SUMMARY ====")
print(f"Raw rows:        {len(df):,}")
print(f"Clean sales rows:{len(df_clean):,}")
print(f"Returns rows:    {len(df_returns):,}")
print(f"Remaining missing values:\n{df_clean.isna().sum()}")


==== CLEANING SUMMARY ====
Raw rows:        1,067,371
Clean sales rows:790,721
Returns rows:    17,879
Remaining missing values:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
Revenue        0
dtype: int64
